In [1]:
import cv2
from python_orb_slam3 import ORBExtractor
import numpy as np
import math

In [2]:
def quat_to_rot(qx, qy, qz, qw):
    q = np.array([qx, qy, qz, qw], dtype=np.float64)
    x, y, z, w = q / np.linalg.norm(q)          # normalize to guard against rounding
    return np.array([
        [1 - 2*(y*y + z*z),     2*(x*y - z*w),     2*(x*z + y*w)],
        [    2*(x*y + z*w), 1 - 2*(x*x + z*z),     2*(y*z - x*w)],
        [    2*(x*z - y*w),     2*(y*z + x*w), 1 - 2*(x*x + y*y)],
    ])

def tum_to_T_cw(tx, ty, tz, qx, qy, qz, qw):
    """TUM pose (camera in world) -> T_cw (world -> camera), 4x4."""
    R_wc = quat_to_rot(qx, qy, qz, qw)
    t_wc = np.array([tx, ty, tz])
    T_cw = np.eye(4)
    T_cw[:3, :3] = R_wc.T
    T_cw[:3, 3] = -R_wc.T @ t_wc
    return T_cw

In [3]:
def orb_bits(hexstr):
  b = bytes.fromhex(hexstr)                       # 32 bytes
  return [(b[i // 8] >> (i % 8)) & 1 for i in range(256)]   # bits[i] = test i

def hamming(h1, h2):
  return bin(int(h1, 16) ^ int(h2, 16)).count("1")

In [4]:
def load_rectification(settings_path):
    """Reproduce ORB-SLAM3 Settings::precomputeRectificationMaps for cam0."""
    fs = cv2.FileStorage(settings_path, cv2.FILE_STORAGE_READ)
    if not fs.isOpened():
        sys.exit(f"Cannot open settings file {settings_path}")
    g = lambda k: fs.getNode(k).real()

    def cam(i):
        K = np.array([[g(f"Camera{i}.fx"), 0, g(f"Camera{i}.cx")],
                      [0, g(f"Camera{i}.fy"), g(f"Camera{i}.cy")],
                      [0, 0, 1]])
        D = np.array([g(f"Camera{i}.k1"), g(f"Camera{i}.k2"), g(f"Camera{i}.p1"), g(f"Camera{i}.p2")])
        return K, D

    K1, D1 = cam(1)
    K2, D2 = cam(2)
    size = (int(g("Camera.width")), int(g("Camera.height")))
    T_c1_c2 = fs.getNode("Stereo.T_c1_c2").mat().astype(np.float64)
    fs.release()

    T_c2_c1 = np.linalg.inv(T_c1_c2)  # ORB-SLAM3 passes Tlr.inverse()
    R12 = np.ascontiguousarray(T_c2_c1[:3, :3])
    t12 = np.ascontiguousarray(T_c2_c1[:3, 3]).reshape(3, 1)
    R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(K1, D1, K2, D2, size, R12, t12,
                                                flags=cv2.CALIB_ZERO_DISPARITY, alpha=-1, newImageSize=size)
    m1, m2 = cv2.initUndistortRectifyMap(K1, D1, R1, P1[:3, :3], size, cv2.CV_32FC1)
    return m1, m2, P1[:3, :3], size

In [5]:
def load_landmarks(path):
    """Returns ids (N,), xyz (N,3), rep_desc (N,32) uint8, all_desc {id: (M,32) uint8}."""
    landmark_dict = {}
    ids, xyz, orb_desc, orb_hex = [],[], [], []
    
    with open(path) as f:
        for line in f:
            t = line.split()
            temp = {}
            if(t[0] == '#'):
                continue
            ids.append(int(t[0]))
            xyz.append( ([float(t[1]), float(t[2]), float(t[3])]) )
            orb_desc.append(np.frombuffer(bytes.fromhex(t[4]), np.uint8))
            orb_hex.append(t[4])
    
    landmark_dict['ids'] = np.array(ids)
    landmark_dict['xyz'] = np.array(xyz)
    landmark_dict['orb_desc'] = np.stack(orb_desc)
    landmark_dict['orb_hex'] = np.stack(orb_hex)

    return landmark_dict

In [65]:
def frustum_filter(xyz, K, width, height, T_cw=np.eye(4), near=0.1, far=np.inf, margin=0.0):
    """
    xyz:   (N,3) landmarks in world (here: first keyframe) coordinates
    K:     (3,3) camera intrinsics
    T_cw:  (4,4) world -> camera transform
    Returns mask (N,) True where the landmark is inside the field of view,
    and uv (N,2) pixel coordinates.
    """
    R, t = T_cw[:3, :3], T_cw[:3, 3]
    p_c = xyz @ R.T + t                      # points in camera frame
    z = p_c[:, 2]

    in_depth = (z > near) & (z < far)        # in front of the camera
    z_safe = np.where(in_depth, z, 1.0)      # avoid dividing by zero or negatives
    uv = (p_c @ K.T)[:, :2] / z_safe[:, None]

    in_image = ((uv[:, 0] >= -margin) & (uv[:, 0] < width + margin) &
                (uv[:, 1] >= -margin) & (uv[:, 1] < height + margin))
    return in_depth & in_image, uv

In [66]:
def get_3Dlandmarks_in_curr_frame(landmark_map, kps, curr_frame_orb_desc, K, width, height,
                                    T_cw, radius=10.0, max_dist=50, ratio=0.8):
    '''
    Projection-guided matching (like ORB-SLAM's SearchByProjection).
    
    For every landmark that projects inside the image, look only at keypoints within
    `radius` pixels of its projection, take the one with the smallest Hamming distance,
    and accept it if dist <= max_dist and it is clearly better than the second-best
    candidate (dist < ratio * second_dist). Each keypoint is assigned to at most one
    landmark (the one with the smallest distance).
    
    landmark_map:        dict from load_landmarks ('ids', 'xyz', 'orb_desc' (N,32) uint8, ...)
    kps:                 list of cv2.KeyPoint from the current rectified image
    curr_frame_orb_desc: (M,32) uint8 descriptors for kps
    K, width, height:    rectified intrinsics and image size
    T_cw:                (4,4) world -> camera pose of the current image
    
    Returns a list of [landmark_idx, landmark_id, kp_idx, hamming_dist, u_kp, v_kp, u_proj, v_proj],
    one entry per matched landmark, sorted by landmark_id.
    '''
    kp_uv = np.array([k.pt for k in kps], dtype=np.float64)          # (M,2)
    kp_bits = np.unpackbits(curr_frame_orb_desc, axis=1)              # (M,256)
    
    # 1. Keep only landmarks in front of the camera and inside the image.
    mask, uv = frustum_filter(landmark_map['xyz'], K, width, height, T_cw=T_cw)
    cand_idx = np.flatnonzero(mask)
    
    # 2. For each candidate landmark, search keypoints near its projection.
    best_for_kp = {}   # kp_idx -> (dist, landmark_idx)
    for li in cand_idx:
      pix = np.hypot(kp_uv[:, 0] - uv[li, 0], kp_uv[:, 1] - uv[li, 1])
      near = np.flatnonzero(pix < radius)
      if near.size == 0:
          continue
    
      lm_bits = np.unpackbits(landmark_map['orb_desc'][li])          # (256,)
      dists = np.count_nonzero(kp_bits[near] != lm_bits, axis=1)     # Hamming distances
    
      order = np.argsort(dists)
      d1 = dists[order[0]]
      if d1 > max_dist:
          continue
      if near.size > 1 and d1 >= ratio * dists[order[1]]:
          continue                                                   # ambiguous match
    
      kp_i = near[order[0]]
      # 3. One landmark per keypoint: keep the closest descriptor.
      if kp_i not in best_for_kp or d1 < best_for_kp[kp_i][0]:
          best_for_kp[kp_i] = (int(d1), li)
    
    landmarks_seen = []
    for kp_i, (d, li) in best_for_kp.items():
      landmarks_seen.append([li, int(landmark_map['ids'][li]), int(kp_i), d,
                             kp_uv[kp_i, 0], kp_uv[kp_i, 1], uv[li, 0], uv[li, 1]])
    landmarks_seen.sort(key=lambda r: r[1])
    return landmarks_seen

In [7]:
# test time: 1403636628763555584.png

imgpath = "/home/shiladitya/Dataset/euroc/MH_01_easy/mav0/cam0/data/1403636639263555584.png"
setting_path = "EuRoC.yaml"
nfeatures = 1200
m1, m2, K, (W, H) = load_rectification(setting_path)
fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]



In [8]:
## Get all landmarks in first camera keyframe coords 
all_landmarks = load_landmarks("landmarks_MH01_stereo_points.txt")

def print_landmark(i):
    print(f"Id:{all_landmarks['ids'][i]}, xyz: {all_landmarks['xyz'][i]}, orb_hex: {all_landmarks['orb_hex'][i]}")

In [12]:
raw = cv2.imread(imgpath)
img = cv2.remap(raw, m1, m2, cv2.INTER_LINEAR)

In [13]:
cv2.imshow('rectified_img', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [55]:
# 2. ORB features (parameters from EuRoC.yaml / ORBextractor.cc)
orb = cv2.ORB_create(nfeatures=nfeatures, scaleFactor=1.2, nlevels=8, edgeThreshold=19,
                     patchSize=31, fastThreshold=20, scoreType=cv2.ORB_FAST_SCORE)

kps, desc = orb.detectAndCompute(img, None)

In [56]:
if desc is None:
    sys.exit("No ORB features detected")
kp_uv = np.array([k.pt for k in kps])

In [67]:
# T_cw from tum

T_cw = tum_to_T_cw(2.910625458, -1.958281159, 5.122889042, -0.025480535, -0.030756075, -0.000350994, 0.999202013)
print(T_cw)

[[ 9.98107881e-01  8.65934686e-04  6.14809524e-02 -3.21838256e+00]
 [ 2.26879036e-03  9.98701238e-01 -5.08988144e-02  2.20988320e+00]
 [-6.14451783e-02  5.09419952e-02  9.96809612e-01 -4.82794239e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [68]:
landmarks_seen = get_3Dlandmarks_in_curr_frame(landmark_map=all_landmarks, kps=kps, curr_frame_orb_desc=desc, K=K, width=W, height=H, T_cw=T_cw)

In [69]:
landmarks_seen

[[18,
  70,
  994,
  24,
  544.9421997070312,
  116.95106506347656,
  544.9653893395454,
  117.96435780343145],
 [47,
  157,
  978,
  39,
  522.5473022460938,
  67.18465423583984,
  522.7640445633368,
  67.79767183590845],
 [75,
  336,
  1180,
  38,
  541.0604858398438,
  114.66181945800781,
  541.6953647027734,
  114.40837615043016],
 [6479,
  1907,
  965,
  44,
  308.5517578125,
  49.76641082763672,
  310.3519483265821,
  50.87914987470363],
 [6480,
  1908,
  871,
  31,
  522.5473022460938,
  70.50241088867188,
  523.1642472207201,
  68.89443760652318],
 [7544,
  14096,
  849,
  27,
  414.7200622558594,
  45.619205474853516,
  415.61450815555935,
  45.33141181532617],
 [6451,
  16203,
  853,
  19,
  327.62884521484375,
  49.76640701293945,
  327.8484581462952,
  50.70082185362116],
 [2048,
  17015,
  1088,
  26,
  653.9306640625,
  373.24810791015625,
  652.682947737514,
  375.2595484414739],
 [2049,
  17019,
  1083,
  18,
  238.8787841796875,
  83.60757446289062,
  240.1103499550152

In [73]:
for lm in landmarks_seen:
    print_landmark(lm[0])

Id:70, xyz: [ 4.68311214 -3.30552554 10.65073013], orb_hex: 78b03b7829fd5147ab5da5181f9f4a3a8993fd101a4bf291dbd9107fe204c663
Id:157, xyz: [ 5.08241272 -4.73322916 13.08060741], orb_hex: 25b495439d6f41997afba958d2714cbb991e40383b74f9128fdf1574c84ac2a0
Id:336, xyz: [ 4.69507694 -3.38719726 10.81039429], orb_hex: 5ab37af8bbfd1957ab5dbdbd8e9fce10a793cd109f2ba2d5dad13affd204877b
Id:1907, xyz: [ 2.09603381 -3.76072288  9.6877985 ], orb_hex: a341674d4c1be4684de0f24a504895a778cc72ebe5f375dc343ac3045d3e7aa4
Id:1908, xyz: [ 5.06569862 -4.6851821  12.99442005], orb_hex: e8ec1d72d9cd55bfb2b5a41837d3401381da48587a6efa37fb8b157d6010c6aa
Id:14096, xyz: [ 3.22085214 -4.54703999 11.58222294], orb_hex: c2607be56c2a78476d80fafb9e1f1452ee955f8108bfc1c096b89a824581e3ab
Id:16203, xyz: [ 2.26981401 -3.75842738  9.68860531], orb_hex: 56a3d6b8cbc138c8af9581a34ccf6c5af79debd112438ad099108befa5072f83
Id:17015, xyz: [ 5.0360322  -0.79974389  8.94154167], orb_hex: fcec9cfe89ed55ceb2b7a019f7fb491bc9d269122a67db33fb

In [75]:
len(landmarks_seen)

155